# Custom Mojo Ops 🚀

> Injecting pure Mojo kernels into the MAX Graph.

In [ ]:
#| default_exp custom_ops

In [ ]:
#| export
from typing import Any, Dict, List, Union
import pyarrow as pa
import pyarrow.compute as pc
import numpy as np
from pathlib import Path
from math import ceil
from max import engine, driver
from max.graph import Graph, TensorType, ops
from max.graph.type import DType, DeviceRef
from mxframe.lazy_expr import Expr
from mxframe.lazy_frame import LogicalPlan, Scan, Filter, Project, Aggregate
from mxframe.compiler import GraphCompiler, _max_dtype

## The `CustomOpsCompiler` Class 🛠️

This class extends `GraphCompiler` to support custom Mojo kernels.

In [ ]:
#| export
# KERNELS_PATH resolves to kernel source directory for on-the-fly compilation.
KERNELS_PATH = (
    str(Path(__file__).parent / "kernels_v261")
    if "__file__" in dir()
    else str(Path("/home/ablearn/mxdf/mxframe/kernels_v261"))
)

WARP_SIZE = 32
MAX_GROUPS = 64
AUTO_GPU_THRESHOLD = 100_000   # rows; used by device='auto' to pick GPU vs CPU


class CustomOpsCompiler(GraphCompiler):
    """Compiles a LogicalPlan into a MAX Graph with custom Mojo kernels.

    Extends GraphCompiler with:
    1. Pre-filter   -- inherited _strip_filters applies Filter nodes in PyArrow.
    2. Group ids    -- PyArrow dictionary-encodes group-by keys into int32 ids.
    3. Kernel dispatch -- grouped sum/min/max/mean/count routed to Mojo kernels.
    4. Keys in result -- group-by key columns are prepended to the output table.
    5. GPU path     -- kernels run on GPU when device='gpu' or device='auto' (N > 100K).
    """

    def __init__(self, kernels_path: str = None, device: str = "cpu"):
        super().__init__(device=device)
        self.kernels_path = kernels_path or KERNELS_PATH

    def _maybe_switch_device(self, N: int) -> None:
        """For device='auto': re-evaluate CPU vs GPU based on row count.

        Switches and re-creates the InferenceSession only when the decision changes.
        Threshold: N > AUTO_GPU_THRESHOLD and GPU available → GPU, otherwise CPU.
        """
        if self._device_arg != "auto":
            return
        want_gpu = self._has_gpu and N > AUTO_GPU_THRESHOLD
        have_gpu = (self._session_device == "gpu")
        if want_gpu == have_gpu:
            return
        if want_gpu:
            self._device_ref = DeviceRef.GPU(0)
            self.session = engine.InferenceSession(devices=[self._gpu_driver])
            self._session_device = "gpu"
        else:
            self._device_ref = DeviceRef.CPU()
            self.session = engine.InferenceSession(devices=[driver.CPU()])
            self._session_device = "cpu"

    def compile_and_run(self, plan) -> pa.Table:
        """Apply filters, build group ids, run custom graph, prepend key columns."""
        plan = self._strip_filters(plan)
        scan = self._find_scan(plan)

        N = scan.table.num_rows
        self._N = N
        self._maybe_switch_device(N)

        extra_inputs = {}
        agg_node = self._find_aggregate(plan)
        group_keys = []
        n_groups = 0
        unique_key_cols = {}

        if agg_node is not None and agg_node.group_by:
            group_keys = [e.args[0] for e in agg_node.group_by]
            group_ids_arr, n_groups, unique_key_cols = self._build_group_ids(
                scan.table, group_keys
            )
            extra_inputs["__group_ids__"] = group_ids_arr.astype(np.int32)

        col_names = []
        col_arrays = {}
        for name in scan.table.column_names:
            arr = np.array(scan.table[name])
            if arr.dtype.kind in ("i", "u", "f", "b"):
                col_names.append(name)
                col_arrays[name] = arr

        all_names = col_names + list(extra_inputs.keys())
        all_arrays = {**col_arrays, **extra_inputs}

        input_types = [
            TensorType(_max_dtype(all_arrays[n]), list(all_arrays[n].shape), self._device_ref)
            for n in all_names
        ]

        graph = Graph(
            name="mxframe_custom",
            input_types=input_types,
            custom_extensions=[Path(self.kernels_path)],
        )
        with graph:
            nodes = {n: graph.inputs[i] for i, n in enumerate(all_names)}
            result_nodes = self._visit_plan_custom(plan, nodes, n_groups=n_groups)
            graph.output(*result_nodes.values())

        model = self.session.load(graph)
        if self._session_device == "gpu":
            execute_inputs = [
                driver.Buffer.from_numpy(all_arrays[n]).to(self._gpu_driver)
                for n in all_names
            ]
        else:
            execute_inputs = [all_arrays[n] for n in all_names]
        output_vals = model.execute(*execute_inputs)

        agg_names = list(result_nodes.keys())
        agg_arrays = [pa.array(np.asarray(t.to_numpy()).reshape(-1)) for t in output_vals]

        if unique_key_cols:
            key_arrays = [unique_key_cols[k] for k in group_keys]
            return pa.Table.from_arrays(key_arrays + agg_arrays, names=group_keys + agg_names)
        return pa.Table.from_arrays(agg_arrays, names=agg_names)

    def _visit_plan_custom(self, plan, nodes, *, n_groups=0):
        if isinstance(plan, Scan):
            return nodes
        elif isinstance(plan, Project):
            upstream = self._visit_plan_custom(plan.input, nodes, n_groups=n_groups)
            out = {}
            for i, expr in enumerate(plan.exprs):
                name = expr._alias or f'col_{i}'
                out[name] = self._visit_expr(expr, upstream)
            return out
        elif isinstance(plan, Filter):
            raise RuntimeError(
                'Filter node reached _visit_plan_custom -- '
                '_strip_filters should have removed all Filter nodes.'
            )
        elif isinstance(plan, Aggregate):
            return self._visit_aggregate_custom(plan, nodes, n_groups=n_groups)
        raise NotImplementedError(f'Unsupported plan node: {type(plan)}')

    def _visit_aggregate_custom(self, plan, nodes, *, n_groups):
        upstream = self._visit_plan_custom(plan.input, nodes, n_groups=n_groups)
        out = {}
        grouped = bool(plan.group_by)
        on_gpu = (self._session_device == "gpu")

        for i, expr in enumerate(plan.aggs):
            name = expr._alias or f'agg_{i}'
            if grouped and n_groups > 0:
                gid_node = ops.cast(upstream["__group_ids__"], DType.int32)

                if on_gpu:
                    if n_groups <= 0:
                        raise ValueError(
                            f"Grouped GPU path requires n_groups > 0; got {n_groups}."
                        )
                    if n_groups > MAX_GROUPS:
                        raise ValueError(
                            f"Grouped GPU path supports up to {MAX_GROUPS} groups per launch; "
                            f"got {n_groups}. Split/chunk groups before GPU execution."
                        )
                    # GPU path: kernels output [num_warps * n_groups] partial results
                    # Python compiler reduces them with ops.reshape + ops.sum/min/max
                    N = getattr(self, '_N', 0)
                    num_warps = ceil(N / WARP_SIZE)
                    if num_warps <= 0:
                        raise ValueError(
                            f"Grouped GPU path requires at least one warp; got N={N}, num_warps={num_warps}."
                        )

                    if expr.op == "count":
                        out_type = TensorType(DType.float32, [num_warps * n_groups], self._device_ref)
                        partial = ops.custom(
                            name="group_count",
                            values=[gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        out[name] = ops.sum(ops.reshape(partial, [num_warps, n_groups]), axis=0)

                    elif expr.op in ("sum", "min", "max"):
                        out_type = TensorType(DType.float32, [num_warps * n_groups], self._device_ref)
                        val_node = ops.cast(self._visit_expr(expr.args[0], upstream), DType.float32)
                        partial = ops.custom(
                            name=f"group_{expr.op}",
                            values=[val_node, gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        reshaped = ops.reshape(partial, [num_warps, n_groups])
                        reduce_fn = ops.sum if expr.op == "sum" else (ops.min if expr.op == "min" else ops.max)
                        out[name] = reduce_fn(reshaped, axis=0)

                    elif expr.op == "mean":
                        # GPU mean = group_sum / group_count (composed, no separate kernel needed)
                        out_type = TensorType(DType.float32, [num_warps * n_groups], self._device_ref)
                        val_node = ops.cast(self._visit_expr(expr.args[0], upstream), DType.float32)
                        sum_partial = ops.custom(
                            name="group_sum",
                            values=[val_node, gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        sum_node = ops.sum(ops.reshape(sum_partial, [num_warps, n_groups]), axis=0)
                        cnt_partial = ops.custom(
                            name="group_count",
                            values=[gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        cnt_node = ops.sum(ops.reshape(cnt_partial, [num_warps, n_groups]), axis=0)
                        out[name] = ops.div(sum_node, ops.cast(cnt_node, DType.float32))

                    else:
                        raise NotImplementedError(f"Grouped '{expr.op}' GPU path not yet implemented.")

                else:
                    # CPU path (unchanged from Phase 1)
                    out_type = TensorType(DType.float32, [n_groups], self._device_ref)
                    if expr.op == "count":
                        out[name] = ops.custom(
                            name="group_count",
                            values=[gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                    elif expr.op in ("sum", "min", "max", "mean"):
                        val_node = ops.cast(self._visit_expr(expr.args[0], upstream), DType.float32)
                        out[name] = ops.custom(
                            name=f"group_{expr.op}",
                            values=[val_node, gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                    else:
                        raise NotImplementedError(
                            f"Grouped '{expr.op}' is not yet supported. "
                            f"Add a group_{expr.op}.mojo kernel and wire it."
                        )
            else:
                out[name] = self._visit_expr(expr, upstream)
        return out

    @staticmethod
    def _find_aggregate(plan):
        if isinstance(plan, Aggregate): return plan
        if hasattr(plan, 'input'): return CustomOpsCompiler._find_aggregate(plan.input)
        return None

    @staticmethod
    def _build_group_ids(table, keys):
        """Dictionary-encode group keys into contiguous int32 group ids."""
        if len(keys) == 1:
            col_arr = table.column(keys[0])
            if isinstance(col_arr, pa.ChunkedArray):
                col_arr = col_arr.combine_chunks()
            encoded = col_arr.dictionary_encode()
            ids = encoded.indices.to_numpy(zero_copy_only=False).astype(np.int32)
            return ids, len(encoded.dictionary), {keys[0]: encoded.dictionary}

        parts = [pc.cast(table.column(k), pa.string()) for k in keys]
        composite = parts[0]
        for p in parts[1:]:
            composite = pc.binary_join_element_wise(composite, p, "|||")
        if isinstance(composite, pa.ChunkedArray):
            composite = composite.combine_chunks()
        encoded = composite.dictionary_encode()
        ids = encoded.indices.to_numpy(zero_copy_only=False).astype(np.int32)
        split_rows = [s.split("|||") for s in encoded.dictionary.to_pylist()]
        unique_key_cols = {keys[i]: pa.array([row[i] for row in split_rows]) for i in range(len(keys))}
        return ids, len(encoded.dictionary), unique_key_cols



## Tests 🧪

Let's verify that our compiler can translate a simple plan and execute it.

In [ ]:
import pyarrow as pa
from mxframe.lazy_expr import col, lit
from mxframe.lazy_frame import LazyFrame, Scan

kernels_path = (
    str(Path(__file__).parent.parent / "mxframe" / "kernels_v261")
    if "__file__" in dir()
    else str(Path("/home/ablearn/mxdf/mxframe/kernels_v261"))
)

compiler = CustomOpsCompiler(kernels_path)

# ── Test 1: Projection falls through to built-in ops ──
table = pa.table({'a': [1, 2, 3], 'b': [4, 5, 6]})
lf = LazyFrame(Scan(table))
result = compiler.compile_and_run(lf.select(col('a') + lit(10)).plan)
assert result.num_columns == 1
assert result.column(0).to_pylist() == [11, 12, 13], f'Projection failed: {result}'
print('✅ Test 1 passed: projection')

# ── Test 2: Global sum via built-in ops ──
table2 = pa.table({'x': [1.0, 2.0, 3.0, 4.0]})
lf2 = LazyFrame(Scan(table2))
result2 = compiler.compile_and_run(
    lf2.groupby().agg(col('x').sum().alias('total')).plan
)
assert abs(result2.column('total').to_pylist()[0] - 10.0) < 1e-6, f'Sum failed: {result2}'
print('✅ Test 2 passed: global sum aggregation')

# ── Test 3: Grouped sum via Mojo group_sum kernel -- with group keys in result ──
table3 = pa.table({
    'group': ['a', 'b', 'a', 'b', 'a'],
    'val':   [1.0, 2.0, 3.0, 4.0, 5.0],
})
lf3 = LazyFrame(Scan(table3))
result3 = compiler.compile_and_run(
    lf3.groupby('group').agg(col('val').sum().alias('total')).plan
)
assert 'group' in result3.column_names, f'Missing group key: {result3.column_names}'
assert result3.column_names[0] == 'group', f'Key should be first: {result3.column_names}'
groups = result3.column('group').to_pylist()
totals = result3.column('total').to_pylist()
result_dict = dict(zip(groups, totals))
assert abs(result_dict['a'] - 9.0) < 1e-6, f'Sum for a should be 9.0: {result_dict}'
assert abs(result_dict['b'] - 6.0) < 1e-6, f'Sum for b should be 6.0: {result_dict}'
print('✅ Test 3 passed: grouped sum with key columns in result (a=9, b=6)')

# ── Test 4: Filter removes rows before grouped aggregation ──
table4 = pa.table({
    'group': ['a', 'b', 'a', 'b', 'a'],
    'val':   [1.0, 2.0, 3.0, 4.0, 5.0],
    'flag':  [1,   1,   0,   1,   1  ],
})
lf4 = LazyFrame(Scan(table4))
result4 = compiler.compile_and_run(
    lf4.filter(col('flag') > lit(0)).groupby('group').agg(col('val').sum().alias('total')).plan
)
groups4 = result4.column('group').to_pylist()
totals4 = result4.column('total').to_pylist()
result_dict4 = dict(zip(groups4, totals4))
assert abs(result_dict4['a'] - 6.0) < 1e-6, f'Filtered sum for a should be 6.0: {result_dict4}'
assert abs(result_dict4['b'] - 6.0) < 1e-6, f'Filtered sum for b should be 6.0: {result_dict4}'
print('✅ Test 4 passed: filter removes rows before grouped aggregation')

# ─── Phase 1 tests ────────────────────────────────────────────────────────────
# Data: group x = [1.0, 2.0], group y = [3.0]
# min(x)=1  max(x)=2  mean(x)=1.5  count(x)=2
# min(y)=3  max(y)=3  mean(y)=3.0  count(y)=1
table5 = pa.table({'g': ['x', 'x', 'y'], 'v': [1.0, 2.0, 3.0]})
lf5 = LazyFrame(Scan(table5))

def _as_dict(result, key_col, val_col):
    return dict(zip(result.column(key_col).to_pylist(), result.column(val_col).to_pylist()))

# ── Test 5: Grouped min ──
r5 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').min().alias('mn')).plan)
d5 = _as_dict(r5, 'g', 'mn')
assert abs(d5['x'] - 1.0) < 1e-5, f'min x should be 1.0: {d5}'
assert abs(d5['y'] - 3.0) < 1e-5, f'min y should be 3.0: {d5}'
print('✅ Test 5 passed: grouped min (x=1.0, y=3.0)')

# ── Test 6: Grouped max ──
r6 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').max().alias('mx')).plan)
d6 = _as_dict(r6, 'g', 'mx')
assert abs(d6['x'] - 2.0) < 1e-5, f'max x should be 2.0: {d6}'
assert abs(d6['y'] - 3.0) < 1e-5, f'max y should be 3.0: {d6}'
print('✅ Test 6 passed: grouped max (x=2.0, y=3.0)')

# ── Test 7: Grouped mean ──
r7 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').mean().alias('avg')).plan)
d7 = _as_dict(r7, 'g', 'avg')
assert abs(d7['x'] - 1.5) < 1e-5, f'mean x should be 1.5: {d7}'
assert abs(d7['y'] - 3.0) < 1e-5, f'mean y should be 3.0: {d7}'
print('✅ Test 7 passed: grouped mean (x=1.5, y=3.0)')

# ── Test 8: Grouped count ──
r8 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').count().alias('cnt')).plan)
d8 = _as_dict(r8, 'g', 'cnt')
assert abs(d8['x'] - 2.0) < 1e-5, f'count x should be 2: {d8}'
assert abs(d8['y'] - 1.0) < 1e-5, f'count y should be 1: {d8}'
print('✅ Test 8 passed: grouped count (x=2, y=1)')

# ── Test 9: Multi-agg groupby (sum + mean in one call) ──
r9 = compiler.compile_and_run(
    lf5.groupby('g').agg(
        col('v').sum().alias('total'),
        col('v').mean().alias('avg'),
    ).plan
)
assert 'g' in r9.column_names and 'total' in r9.column_names and 'avg' in r9.column_names
d9_sum  = _as_dict(r9, 'g', 'total')
d9_mean = _as_dict(r9, 'g', 'avg')
assert abs(d9_sum['x']  - 3.0) < 1e-5, f'sum x should be 3.0: {d9_sum}'
assert abs(d9_mean['x'] - 1.5) < 1e-5, f'mean x should be 1.5: {d9_mean}'
print('✅ Test 9 passed: multi-agg groupby (sum + mean)')

print('\nAll CustomOpsCompiler tests passed! 🎉')


✅ Test 1 passed: projection
✅ Test 2 passed: global sum aggregation
✅ Test 3 passed: grouped sum with key columns in result (a=9, b=6)
✅ Test 4 passed: filter removes rows before grouped aggregation
✅ Test 5 passed: grouped min (x=1.0, y=3.0)
✅ Test 6 passed: grouped max (x=2.0, y=3.0)
✅ Test 7 passed: grouped mean (x=1.5, y=3.0)
✅ Test 8 passed: grouped count (x=2, y=1)
✅ Test 9 passed: multi-agg groupby (sum + mean)

All CustomOpsCompiler tests passed! 🎉
